# 02 · Smoke test: Naive, MA(5), ARIMA

Same models as the midterm, now run through the **shared walk-forward code** (`src/fincare_common.py`). Two checks:
1. RMSE matches the midterm (Naive 0.966, MA(5) 1.073, ARIMA 0.964; difference > 0.01 = bug).
2. Training time and latency (ms per day) print for every model.

*Edited from the midterm baselines notebook (kept in `notebooks/archive_midterm/`).*

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
import config as C, baselines as B, make_results
from fincare_common import load_master, walk_forward
MASTER = os.environ.get("MASTER_PATH", str(C.MASTER_PATH))
OUT = os.environ.get("RESULTS_OUT", str(C.OUT_DIR))
df = load_master(MASTER); df.shape

In [ ]:
for name, fit, pred in [("naive", B.naive_fit, B.naive_predict), ("ma5", B.ma_fit, B.ma_predict), ("arima", B.arima_fit, B.arima_predict)]:
    walk_forward(fit, pred, name, df=df, out_dir=OUT)

## Results (all test days pooled; RMSE and MAE in % points)

In [ ]:
tabs = make_results.main(OUT)
tabs['results_table.csv']

## Check 1: compare with the midterm

In [ ]:
MIDTERM = {"Naive (return = 0)": 0.966, "Moving average (5-day)": 1.073, "ARIMA": 0.964}
t = tabs["results_table.csv"].set_index("Model")
{m: (round(float(t.loc[m, "RMSE (%)"]), 3), v, abs(t.loc[m, "RMSE (%)"] - v) <= 0.01) for m, v in MIDTERM.items()}
# (now, midterm, within 0.01?)  -- on fake data these differ, ignore. The full pass/fail check is `python src/smoke_test.py`.

## Check 2: timers
Train time and latency must be filled in for every model (see table above).

In [ ]:
assert t.loc["ARIMA", "Train time per fold (s)"] > 0 and (t["Inference per day (ms)"].dropna() > 0).all()
print("Timers OK")

## Per fold (Table A) and without the 5 biggest-move days (Table B)

In [ ]:
tabs['table_A_by_fold.csv']

In [ ]:
tabs['table_B_no_big_days.csv']

In [ ]:
tabs['calm_vs_stressed.csv']